In [ ]:
!pip install -q open-clip-torch tqdm

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/liya_diplomCC')

DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'

In [ ]:
from scripts.verify_dataset import compute_clip_scores
import matplotlib.pyplot as plt
import numpy as np

scores = compute_clip_scores(
    f'{DRIVE_ROOT}/data/captioned_pairs.jsonl',
    sample_size=200,
    seed=42,
)
print(f"CLIP Score — mean={np.mean(scores):.3f}, "
      f"min={np.min(scores):.3f}, max={np.max(scores):.3f}")
print(f"Pairs above 0.25: {sum(s >= 0.25 for s in scores)}/200")

plt.hist(scores, bins=30, color='steelblue', edgecolor='white')
plt.axvline(0.25, color='red', linestyle='--', label='threshold=0.25')
plt.xlabel("CLIP Score")
plt.ylabel("Count")
plt.title("CLIP Score distribution (200 sampled pairs)")
plt.legend()
plt.savefig(f'{DRIVE_ROOT}/results/metrics/clip_score_distribution.png', dpi=150)
plt.show()

In [ ]:
import json, random

random.seed(42)
with open(f'{DRIVE_ROOT}/data/captioned_pairs.jsonl') as f:
    all_pairs = [json.loads(l) for l in f]

random.shuffle(all_pairs)

test_500  = all_pairs[:500]
train_2k  = all_pairs[500:2500]
train_10k = all_pairs[500:10500]

def save_split(pairs, path, split_name):
    with open(path, 'w') as f:
        for item in pairs:
            item['split'] = split_name
            f.write(json.dumps(item) + '\n')
    print(f"Saved {len(pairs)} pairs \u2192 {path}")

save_split(test_500,  f'{DRIVE_ROOT}/data/test_500.jsonl',  'test')
save_split(train_2k,  f'{DRIVE_ROOT}/data/train_2k.jsonl',  'train_2k')
save_split(train_10k, f'{DRIVE_ROOT}/data/train_10k.jsonl', 'train_10k')

In [ ]:
import json

for name, path, expected in [
    ('test_500',  f'{DRIVE_ROOT}/data/test_500.jsonl',  500),
    ('train_2k',  f'{DRIVE_ROOT}/data/train_2k.jsonl',  2000),
    ('train_10k', f'{DRIVE_ROOT}/data/train_10k.jsonl', 10000),
]:
    with open(path) as f:
        items = [json.loads(l) for l in f]
    print(f"{name}: {len(items)} pairs (expected {expected})")
    assert len(items) == expected, f"Size mismatch for {name}"

# Verify no overlap between test and train
test_paths = {json.loads(l)['png_path']
              for l in open(f'{DRIVE_ROOT}/data/test_500.jsonl')}
train_paths = {json.loads(l)['png_path']
               for l in open(f'{DRIVE_ROOT}/data/train_2k.jsonl')}
assert len(test_paths & train_paths) == 0, "OVERLAP between test and train!"
print("PASS: no overlap between test and train splits")